ARIMA - LTC/USD
Python 3.11.8

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm # For auto_arima
import plotly.graph_objects as go

1. Data Loading and Preparation

In [11]:
# Download Bitcoin data
df_LTC = yf.download(
    tickers=["LTC-USD"],
    start="2020-01-01",
    end="2025-01-01" 
)

# Basic Cleaning and Selection
df_LTC.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
print(f"Original shape: {df_LTC.shape}")
print('Null Values Before:', df_LTC.isnull().values.sum())
# Forward fill common for financial data if few NaNs exist, or drop
df_LTC.ffill(inplace=True) 
print('Null Values After FFill:', df_LTC.isnull().values.sum())

# Select Target and Set Index
df_LTC.reset_index(inplace=True)
df_LTC['Date'] = pd.to_datetime(df_LTC['Date'], format='%Y-%m-%d')
df_LTC = df_LTC[['Date', 'Close']]
df_LTC.set_index('Date', inplace=True)

# Ensure Daily Frequency
df_LTC = df_LTC.asfreq('D')
# Re-check for NaNs introduced by asfreq
print('Null Values After AsFreq:', df_LTC.isnull().values.sum())
df_LTC.ffill(inplace=True) 
print(f"Shape after preproc: {df_LTC.shape}")
print(f"Frequency of the index: {df_LTC.index.freq}")
print("\nData Head:")
print(df_LTC.head())

[*********************100%***********************]  1 of 1 completed

Original shape: (1827, 5)
Null Values Before: 0
Null Values After FFill: 0
Null Values After AsFreq: 0
Shape after preproc: (1827, 1)
Frequency of the index: <Day>

Data Head:
                Close
Date                 
2020-01-01  41.326534
2020-01-02  42.018085
2020-01-03  39.863129
2020-01-04  42.383526
2020-01-05  43.291382


2. Data Splitting

In [12]:
# Define split point (e.g., 80% train, 20% test)
train_size = int(len(df_LTC) * 0.80)
train_data = df_LTC[:train_size]
test_data = df_LTC[train_size:]

print(f"\nTraining Data: {len(train_data)} points ({train_data.index.min()} to {train_data.index.max()})")
print(f"Test Data:     {len(test_data)} points ({test_data.index.min()} to {test_data.index.max()})")

# Define Forecast Horizon
fh = len(test_data)


Training Data: 1461 points (2020-01-01 00:00:00 to 2023-12-31 00:00:00)
Test Data:     366 points (2024-01-01 00:00:00 to 2024-12-31 00:00:00)


3. Stationarity Check

In [13]:
# We expect LTC prices to be non-stationary, requiring differencing (d > 0).
def check_stationarity(timeseries):
    print("\nResults of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)
    if dftest[1] <= 0.05:
        print("=> Conclusion: Data is likely Stationary (reject H0)")
    else:
        print("=> Conclusion: Data is likely Non-Stationary (fail to reject H0)")

print("\n--- Stationarity Check on Training Data ---")
check_stationarity(train_data['Close'])


--- Stationarity Check on Training Data ---

Results of Dickey-Fuller Test:
Test Statistic                   -1.887279
p-value                           0.338069
#Lags Used                       15.000000
Number of Observations Used    1445.000000
Critical Value (1%)              -3.434884
Critical Value (5%)              -2.863542
Critical Value (10%)             -2.567836
dtype: float64
=> Conclusion: Data is likely Non-Stationary (fail to reject H0)


4. ARIMA Model Building (using auto_arima)

In [14]:
print("\n--- Running auto_arima ---")
# adjusted 'm' to m=7 for daily data with weekly seasonal ARIMA (assumption)
auto_model = pm.auto_arima(train_data['Close'],
                           start_p=1, start_q=1,
                           test='adf',        
                           max_p=3, max_q=3,  
                           m=7,               
                           start_P=0, seasonal=True,  
                           d=None,           
                           D=None,            
                           trace=True,        
                           error_action='ignore',
                           suppress_warnings=True,
                           stepwise=True)     

print("\n--- Best Model Found ---")
print(auto_model.summary())

# Extract the best model order found
print(f"\nBest ARIMA Order: {auto_model.order}")
print(f"Best Seasonal Order: {auto_model.seasonal_order}")


--- Running auto_arima ---
Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,0,1)[7] intercept   : AIC=9886.543, Time=5.88 sec
 ARIMA(0,1,0)(0,0,0)[7] intercept   : AIC=9904.156, Time=0.11 sec
 ARIMA(1,1,0)(1,0,0)[7] intercept   : AIC=9897.557, Time=3.10 sec
 ARIMA(0,1,1)(0,0,1)[7] intercept   : AIC=9898.089, Time=2.59 sec
 ARIMA(0,1,0)(0,0,0)[7]             : AIC=9902.170, Time=0.07 sec
 ARIMA(1,1,1)(0,0,0)[7] intercept   : AIC=9885.357, Time=0.81 sec
 ARIMA(1,1,1)(1,0,0)[7] intercept   : AIC=9886.715, Time=5.02 sec
 ARIMA(1,1,1)(1,0,1)[7] intercept   : AIC=9885.583, Time=11.39 sec
 ARIMA(0,1,1)(0,0,0)[7] intercept   : AIC=9897.740, Time=0.30 sec
 ARIMA(1,1,0)(0,0,0)[7] intercept   : AIC=9896.848, Time=0.19 sec
 ARIMA(2,1,1)(0,0,0)[7] intercept   : AIC=9886.670, Time=1.19 sec
 ARIMA(1,1,2)(0,0,0)[7] intercept   : AIC=9886.564, Time=1.92 sec
 ARIMA(0,1,2)(0,0,0)[7] intercept   : AIC=9896.269, Time=1.20 sec
 ARIMA(2,1,0)(0,0,0)[7] intercept   : AIC=9894.832, Time=0.71 sec
 ARI

5. Forecasting on Test Set

In [15]:
# Generate predictions for the forecast horizon (length of test set)
# Use the fitted auto_arima model
predictions_arima = auto_model.predict(n_periods=fh)

# Create a pandas Series for the predictions with the correct index
predictions_arima = pd.Series(predictions_arima, index=test_data.index)

print("\n--- ARIMA Predictions (first 5) ---")
print(predictions_arima.head())


--- ARIMA Predictions (first 5) ---
Date
2024-01-01    73.162489
2024-01-02    73.271051
2024-01-03    73.188843
2024-01-04    73.251094
2024-01-05    73.203955
Freq: D, dtype: float64


6. Model Evaluation

In [19]:
# Calculate metrics
y_true = test_data['Close']
y_pred = predictions_arima

rmse_arima = np.sqrt(mean_squared_error(y_true, y_pred))
mae_arima = mean_absolute_error(y_true, y_pred)
mape_arima = mean_absolute_percentage_error(y_true, y_pred)
r2_arima = r2_score(y_true, y_pred) # R-squared can be negative for poor models

# Print Evaluation Metrics
print("\n--- ARIMA Model Evaluation Metrics (Hold-out Set) ---")
print(f"Root Mean Squared Error (RMSE): {rmse_arima:.4f}")
print(f"Mean Absolute Error (MAE):   {mae_arima:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape_arima:.4%}")
print(f"R-squared (R²):              {r2_arima:.4f}") # R² interpretation depends on context

# Print Metrics for Comparison Table in Thesis
print("\n--- ARIMA Hold-out Metrics (for LTC) ---")
print(f"ARIMA Hold-out RMSE:  {rmse_arima:.4f}")
print(f"ARIMA Hold-out MAE:   {mae_arima:.4f}")
print(f"ARIMA Hold-out MAPE:  {mape_arima:.4%}")
print(f"ARIMA Hold-out R²:    {r2_arima:.4f}")


--- ARIMA Model Evaluation Metrics (Hold-out Set) ---
Root Mean Squared Error (RMSE): 16.4750
Mean Absolute Error (MAE):   11.3642
Mean Absolute Percentage Error (MAPE): 12.9617%
R-squared (R²):              -0.1245

--- ARIMA Hold-out Metrics (for LTC) ---
ARIMA Hold-out RMSE:  16.4750
ARIMA Hold-out MAE:   11.3642
ARIMA Hold-out MAPE:  12.9617%
ARIMA Hold-out R²:    -0.1245


7. Visualization

In [22]:
# Prepare data for plotting
plot_train = train_data.reset_index()
plot_test = test_data.reset_index()
plot_pred = pd.DataFrame({'Date': predictions_arima.index, 'Predictions': predictions_arima.values})

# Create Plotly figure
fig = go.Figure()

# Add traces
fig.add_trace(go.Scatter(x=plot_train['Date'], y=plot_train['Close'],
                         mode='lines', name='Actual Price (Train)',
                         line=dict(color='blue')))
fig.add_trace(go.Scatter(x=plot_test['Date'], y=plot_test['Close'],
                         mode='lines', name='Actual Price (Test)',
                         line=dict(color='green')))
fig.add_trace(go.Scatter(x=plot_pred['Date'], y=plot_pred['Predictions'],
                         mode='lines', name='ARIMA Predictions',
                         line=dict(color='red', dash='dash')))

# Update layout
fig.update_layout(
    title="Bitcoin Price Forecasting using ARIMA",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    legend_title="Legend",
    template="plotly_white"
)

# Show plot
fig.show()

8. Data Split Summary

In [18]:
print("\n--- Data Split Summary ---")
train_start_date = train_data.index.min().strftime('%Y-%m-%d')
train_end_date = train_data.index.max().strftime('%Y-%m-%d')
test_start_date = test_data.index.min().strftime('%Y-%m-%d')
test_end_date = test_data.index.max().strftime('%Y-%m-%d')

print(f"Training Set: {len(train_data)} data points from {train_start_date} to {train_end_date}")
print(f"Test Set:     {len(test_data)} data points from {test_start_date} to {test_end_date}")



--- Data Split Summary ---
Training Set: 1461 data points from 2020-01-01 to 2023-12-31
Test Set:     366 data points from 2024-01-01 to 2024-12-31
